# **1) Download the necessary Libraries**

In [1]:
!pip install -U transformers datasets peft trl accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# **2) Upload the Dataset files (cleaned_train and cleaned_valid)**

In [2]:
from google.colab import files

# This will open a browser upload dialog window
uploaded = files.upload()

# Confirming successful upload
for filename in uploaded.keys():
    print(f'Successfully uploaded file "{filename}" with length {len(uploaded[filename])} bytes')

Saving cleaned_train.jsonl to cleaned_train.jsonl
Saving cleaned_valid.jsonl to cleaned_valid.jsonl
Successfully uploaded file "cleaned_train.jsonl" with length 142065 bytes
Successfully uploaded file "cleaned_valid.jsonl" with length 15808 bytes


# **3) Loading the Models and trainings**

In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import os

# =========================
# MODEL
# =========================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# =========================
# DOWNLOADS FOLDER PATH
# =========================

downloads_folder = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "qwen_model"
)

# Create folder if it doesn't exist
os.makedirs(downloads_folder, exist_ok=True)

print(f"Model will be downloaded to: {downloads_folder}")

# =========================
# LOAD DATASET
# =========================

dataset = load_dataset(
    "json",
    data_files={
        "train": "cleaned_train.jsonl",
        "validation": "cleaned_valid.jsonl"
    }
)

# =========================
# TOKENIZER
# =========================

print("Downloading/loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=downloads_folder,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

# =========================
# 4-BIT CONFIG
# =========================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# =========================
# LOAD MODEL
# =========================

print("Downloading/loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=downloads_folder,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")

model.config.use_cache = False

# =========================
# LORA CONFIG
# =========================

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# =========================
# TRAINING ARGUMENTS
# =========================

training_args = TrainingArguments(
    output_dir="./socratic-model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none"
)

# =========================
# TRAINER
# =========================

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    args=training_args,
)

# =========================
# TRAIN
# =========================

print("Starting training...")

trainer.train()

# =========================
# SAVE TRAINED MODEL
# =========================

save_folder = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "socratic-model"
)

os.makedirs(save_folder, exist_ok=True)

print("Saving trained model...")

trainer.model.save_pretrained(save_folder)
tokenizer.save_pretrained(save_folder)

print("Training complete!")
print(f"Downloaded model cache: {downloads_folder}")
print(f"Trained model saved at: {save_folder}")

Model will be downloaded to: /root/Downloads/qwen_model


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Downloading/loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
10,3.842013
20,2.736417
30,1.831536
40,1.216786
50,0.881195
60,0.613667
70,0.429257
80,0.358369
90,0.327699
100,0.305112


Saving trained model...
Training complete!
Downloaded model cache: /root/Downloads/qwen_model
Trained model saved at: /root/Downloads/socratic-model


# **4) Install the Package required for Ngrok hosting**

In [4]:
# 1. Install required packages for hosting (With the torchao upgrade fix!)
!pip install flask-ngrok pyngrok flask-cors torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.0 MB/s eta 0:00:00


# **5) The Netwroking Tunnel which creates a link used in the pycharm project to access and send data**

In [5]:
# 1. Install required packages for hosting
!pip install flask-ngrok pyngrok flask-cors -q

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

app = Flask(__name__)
CORS(app) # Allows external traffic from PyCharm

# 2. Authenticate ngrok (Get your free token from dashboard.ngrok.com)
# Sign up for a free account on ngrok and paste your token inside the quotes below!
NGROK_TOKEN = "3EEjSx55H0waGw8iZVJjSBXsQwM_36vVN8nXgyNwu9gkgFRfV"
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Load the model cleanly using Colab's fast VRAM
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.float16, device_map="auto")

# Point this directly to your final checkpoint directory in Colab!
model = PeftModel.from_pretrained(base_model, "./socratic-model/checkpoint-171")
model.eval()

@app.route("/generate", methods=["POST"])
def generate():
    data = request.json
    user_input = data.get("prompt", "")

    # NEW GROUNDED SYSTEM PROMPT: Gives the model strict guardrails
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful, logical Socratic computer science tutor. "
                "Your goal is to guide students to the answer by asking informative hints. "
                "Do not repeat the same question twice. If the user asks something completely "
                "unrelated to programming, answer it normally and politely without being Socratic."
            )
        },
        {"role": "user", "content": user_input}
    ]

    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.6,          # Lowered slightly (0.7 -> 0.6) to make it more focused/less chaotic
            top_p=0.85,               # Keeps it tracking high-probability, logical words
            repetition_penalty=1.2,   # CRITICAL: Punishes the model mathematically if it tries to repeat or loop!
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return jsonify({"response": response_text})

# 4. Open the Tunnel and Run
public_url = ngrok.connect(5000)
print(f"\n🚀 COPY THIS URL AND PASTE IT INTO PYCHARM:\n{public_url}\n")

app.run()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


🚀 COPY THIS URL AND PASTE IT INTO PYCHARM:
NgrokTunnel: "https://emu-unsnap-dried.ngrok-free.dev" -> "http://localhost:5000"

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
